# Imports and set-up

In [1]:
import ase.io

import sys
import os
import glob
import subprocess
import json

In [2]:

current_folder = os.getcwd()

# 1. Get the path to the directory two levels up
parent = os.path.abspath(os.path.join(current_folder, ".."))

# 2. Build the full path to the internal folder
target_dir = os.path.join(parent, "tabgap_install", "tabgap", "tabgap")

# 3. Add to sys.path
if target_dir not in sys.path:
    sys.path.append(target_dir)

# 4. Imports
from gap_fit import *
from gap_fit_params_default import *

In [3]:
# Multiple species
db = ase.io.read('/Users/jegorsbalzins/jgap/resources/xyz-samples/feni-train.xyz', index=':')

desc_copy = None

# Validate regression coefficients

### Use tabgap's default descriptor params, but smaller

In [4]:

if desc_copy is None:
    desc_copy = descriptors

descriptors = desc_copy

if 'soap' in descriptors:
    del descriptors['soap']


if 'distance_2b' in descriptors:
    descriptors['distance_2b']['n_sparse'] = 5
    #del descriptors['distance_2b']
    pass

if 'eam_density' in descriptors:
    descriptors['eam_density']['n_sparse'] = 10
    #del descriptors['eam_density']
    pass
if 'angle_3b' in descriptors:
    descriptors['angle_3b']['n_sparse'] = 10
    #del descriptors['angle_3b']
    pass
descriptors_vec = [v for k, v in descriptors.items()]
descriptors_vec

db_cpy = None


In [5]:
# ? 'config_type_sigma': '{isolated_atom:0.0001:0.04:0.01:0.0:liquid:0.01:0.5:2.0:0.0:liquid_composition:0.01:0.5:2.0:0.0:liquid_hea:0.01:0.5:2.0:0.0:surf_liquid:0.01:0.4:0.2:0.0:dimer:0.1:1.0:1.0:0.0:short_range:0.05:0.8:0.8:0.0:hea_short_range:0.05:0.8:2.0:0.0:hea_small:0.01:0.1:0.5:0.0:composition:0.01:0.1:0.5:0.0:binary_alloys:0.01:0.1:0.5:0.0:hea_vacancies:0.01:0.1:0.5:0.0:hea_ints:0.01:0.1:0.5:0.0:hea_vac_saddle:0.01:0.1:0.5:0.0}',

global_args

{'sparse_jitter': '1e-8',
 'do_copy_at_file': 'False',
 'gp_file': 'gap.xml',
 'rnd_seed': '999',
 'default_sigma': '{0.002 0.1 0.2 0.5}'}

## Run QUIP GAP fit
### Setup env variable used by tabgap util

In [6]:
# Your base path
base_build_dir = os.path.join(parent, "quip_install", "QUIP", "build")

# Use glob to find the directory inside (the * matches the one dir)
# Result is a list, so we take the first element [0]
target_dir = glob.glob(os.path.join(base_build_dir, "*"))[0]

# Now join 'quip' to that resolved path
final_path = os.path.join(target_dir, "gap_fit")

print(final_path)

os.environ.setdefault('GAP_FIT', final_path)

/Users/jegorsbalzins/jgap/benchmarking/quip_install/QUIP/build/darwin_x86_64_gfortran/gap_fit


'/Users/jegorsbalzins/jgap/benchmarking/quip_install/QUIP/build/darwin_x86_64_gfortran/gap_fit'

### Run

In [7]:
if db_cpy is None:
    db_cpy = db

db = db_cpy

for i in range(len(db)):
    a = db[i]
    #if 'trimer' in a.info['config_type']:
    if 'short' in a.info['config_type']:
        #print(i)
        #print(a)
        pass
    if 'virial' in a.info:
        #del a.info['virial']
        pass
    if 'Ni' not in a.symbols:
        #print(i)
        pass

to_be_fit = [db[0], db[1], db[34]]
#to_be_fit = [db[1], db[301]]
print(to_be_fit)
#del to_be_fit[0].arrays['force']
#del to_be_fit[1].arrays['force']

[Atoms(symbols='Ni', pbc=True, cell=[16.0, 17.0, 18.0], force=..., calculator=SinglePointCalculator(...)), Atoms(symbols='Fe', pbc=True, cell=[20.840052, 20.840052, 20.840052], force=..., calculator=SinglePointCalculator(...)), Atoms(symbols='FeNi2FeNi4Fe2NiFe2Ni2FeNi2Fe4NiFe2NiFeNiFeNiFeNi3Fe2NiFe2NiFeNi4FeNi2Fe', pbc=True, cell=[7.204231749195973, 7.204231749195973, 10.806347623793991], force=..., calculator=SinglePointCalculator(...))]


In [8]:

run(to_be_fit, global_args, descriptors_vec, compute_errors=False, rundir='quip-out')